# Deliverable 4 — Independent pseudo-spectral comparator

Benchmarks a dealiased Fourier-vorticity RK4 solver against D2Q9 BGK at matched physical parameters.

This notebook is an executable evidence artifact. Its default configuration is
deliberately small enough for a clean local rerun; scale-up parameters are
listed separately and are not represented as measured results.

In [1]:
from pathlib import Path
import sys

repo_root = Path.cwd().parent if Path.cwd().name == "deliverables" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
output_dir = repo_root / "results" / "deliverables"
output_dir.mkdir(parents=True, exist_ok=True)

## Comparator protocol

Both solvers use the canonical periodic box, the same final time, Reynolds
number, grid, and relative velocity L2 definition. The spectral solver is an
independent discretization; its timings include time integration but exclude
plotting and file output, matching the LBM timing boundary.

In [2]:
import numpy as np
import pandas as pd
from quantum_aero.classical import LBMConfig, run_lbm
from quantum_aero.deliverables import run_pseudospectral_tgv

rows = []
for reynolds in (10, 100, 1000):
    for n in (16, 32, 64):
        cfg = LBMConfig(n=n, reynolds=reynolds, t_end=0.25, mach=0.05, snapshots=2)
        spectral = run_pseudospectral_tgv(cfg, repeats=3)
        lbm_trials = [run_lbm(cfg) for _ in range(3)]
        lbm_final = lbm_trials[-1]["records"][-1]
        rows.extend([
            {"solver": "Fourier-vorticity RK4", "reynolds": reynolds, "n": n,
             "runtime_median_seconds": spectral["runtime_median_seconds"], "relative_l2": spectral["relative_l2"],
             "steps": spectral["steps"], "memory_bytes": spectral["memory_bytes"]},
            {"solver": "D2Q9 BGK", "reynolds": reynolds, "n": n,
             "runtime_median_seconds": float(np.median([x["runtime_seconds"] for x in lbm_trials])),
             "relative_l2": lbm_final["relative_l2"], "steps": lbm_trials[-1]["steps"],
             "memory_bytes": lbm_trials[-1]["population_memory_bytes"]},
        ])
df = pd.DataFrame(rows)
df.to_csv(output_dir / "04_spectral_comparator.csv", index=False)
df

,solver,reynolds,n,runtime_median_seconds,relative_l2,steps,memory_bytes
0,Fourier-vorticity RK4,10,16,0.021090,3.892515e-09,6,24576
1,D2Q9 BGK,10,16,0.059019,1.396138e-02,50,18432
2,Fourier-vorticity RK4,10,32,0.045554,3.434779e-10,11,98304
3,D2Q9 BGK,10,32,0.183438,3.942512e-03,99,73728
4,Fourier-vorticity RK4,10,64,0.163448,2.581148e-11,21,393216
5,D2Q9 BGK,10,64,1.083538,2.010362e-03,198,294912
6,Fourier-vorticity RK4,100,16,0.019989,3.619357e-09,6,24576
7,D2Q9 BGK,100,16,0.060763,1.195712e-02,50,18432
8,Fourier-vorticity RK4,100,32,0.050832,3.202830e-10,11,98304
9,D2Q9 BGK,100,32,0.194797,4.421642e-03,99,73728


In [3]:
assert len(df) == 18
spectral_max = df[df.solver == "Fourier-vorticity RK4"].relative_l2.max()
print("PASS: independent comparator completed; maximum spectral L2 error =", spectral_max)
print("Median results by solver:")
df.groupby("solver")[["runtime_median_seconds", "relative_l2"]].median()

PASS: independent comparator completed; maximum spectral L2 error = 3.89251481520151e-09
Median results by solver:


,runtime_median_seconds,relative_l2
solver,,
D2Q9 BGK,0.194797,4.421642e-03
Fourier-vorticity RK4,0.050635,3.208198e-10


## Interpretation boundary

This TGV is a special single-mode solution, so the comparator is expected to
be exceptionally strong. The result is the correct baseline for this challenge
instance, not a general performance claim for arbitrary turbulent CFD.